### Import libraries

In [1]:
import time
from pprint import pprint

import grain.python as grain

import jax
import jax.numpy as jnp

import flax.nnx as nnx
import optax

from datasets import load_dataset

from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
# from tokenizers.pre_tokenizers import Whitespace

/home/arjunprasaath/miniconda3/envs/llm_jax/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Architecture

#### Things to build:

- [ x ] Load the dataset
- [ x ] Preprocess the dataset
- [ x ] Dataloader for training (https://google-grain.readthedocs.io/en/latest/tutorials/data_sources/huggingface_dataset_tutorial.html)
- [ x ] Building a Tokenizer (https://huggingface.co/docs/tokenizers/en/quicktour?code=python)
- [ x ] Build a Token embedding and Positional embedding
- [ x ] Build Transformer block
    - [ x ] MHA block
    - [ x ] Feed-Forward block
- [ x ] Ouput block
- [ x ] Generate samples/ output
- [ x ] Training script
- [ x ] Testing script




#### Load the dataset

In [2]:
ds = load_dataset("karpathy/tinystories-gpt4-clean", split="train")

In [3]:
test = ds.select(range(0, 100_000))

In [5]:
test['text'][0]

'Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.\nJack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.\nOnce the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!\nThe more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that.'

In [6]:
# DONT RUN THIS (runs over the 500_000 samples to check the max number of tokens in those samples in a story)
import numpy as np

# Sample 500,000 random stories
sample = ds.shuffle(seed=42).select(range(500_000))
encodings = tokenizer.encode_batch(sample["text"])
lengths = [len(e.ids) for e in encodings]

print(f"Max in sample:    {np.max(lengths)}")
print(f"95th percentile:  {np.percentile(lengths, 95)}")
print(f"99th percentile:  {np.percentile(lengths, 99)}")
print(f"99.9th percentile:{np.percentile(lengths, 99.9)}")


Max in sample:    706
95th percentile:  230.0
99th percentile:  354.0
99.9th percentile:519.0


#### Building the tokenizer

In [54]:
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
trainer = BpeTrainer(special_tokens=["[PAD]", "[UNK]", "<|startoftext|>", "<|endoftext|>"], vocab_size=10000)


In [55]:
def batch_iterator(batch_size=1):
    tok_dataset = test.select_columns("text")
    for batch in tok_dataset.iter(batch_size):
        yield batch["text"]

In [56]:
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer, length=len(test))

In [57]:
tokenizer.save("data/tokenizer.json")

#### Preprocess the dataset

In [25]:
# adding <|startoftext|> and <|endoftext|> tokens to the stories.
def add_spl_tokens(example):
    example['text'] = "<|startoftext|> " + example['text'] + " <|endoftext|>"
    return example

In [32]:
updated_ds = test.map(add_spl_tokens)

In [31]:
print(updated_ds["text"][0])

<|startoftext|> Once there was a little boy named Jack. He was only three years old and had lots of things he wanted to do. One day he saw something very special in the park - a big wheel! It was big and bright and looked very inviting.
Jack wanted to get on the wheel, so he ran to it. He put his hands on it and gently started turning it around. The wheel spun faster and he laughed as he felt the wind blowing in his face.
Once the wheel slowed down, Jack got off and looked around. He had such a great time on the wheel. He put his hands back on it and started turning it again, and soon he was spinning faster and faster. Every time he got off the wheel, he would put his hands back on it and start turning it again!
The more Jack put his hands on this big, bright wheel and gently turned it, the more he loved it. It was so much fun and he would happily keep turning it every day after that. <|endoftext|>


#### Dataloader

In [4]:
maxlen = 1024

tokenizer = Tokenizer.from_file("data/tokenizer.json")
pad_id = tokenizer.token_to_id("[PAD]")
tokenizer.enable_padding(length=maxlen, pad_id = pad_id, pad_token="[PAD]")
tokenizer.enable_truncation(max_length=maxlen)

In [6]:
# adding <|startoftext|> and <|endoftext|> tokens to the stories.
def add_spl_tokens(example):
    example['text'] = "<|startoftext|> " + example['text'] + " <|endoftext|>"
    return example

def encoding_text(example):
    return tokenizer.encode(example['text']).ids

In [6]:
# Have a look at: https://google-grain.readthedocs.io/en/latest/tutorials/data_sources/huggingface_dataset_tutorial.html
dataset = (
    grain.MapDataset.source(ds)
    .shuffle(seed=40)
    .map(add_spl_tokens)
    .map(encoding_text)
    .map(jnp.asarray)
    .batch(batch_size=3)
)

In [9]:
for i in dataset:
    pprint(i)
    break

array([[   2,    5,  326, ...,    0,    0,    0],
       [   2,    5, 4636, ...,    0,    0,    0],
       [   2,    5, 2272, ...,    0,    0,    0]],
      shape=(3, 1024), dtype=int32)


In [7]:
# Have a look at : https://google-grain.readthedocs.io/en/latest/tutorials/data_loader_tutorial.html#transformations
class ProcessStory(grain.MapTransform):
    def map(self, element):
        text = "<|startoftext|> " + element['text'] + " <|endoftext|>"
        tokenized_text = tokenizer.encode(text).ids
        return jnp.asarray(tokenized_text)

sampler = grain.IndexSampler(
    num_records = len(test),
    num_epochs = 1,
    shuffle = True,
    seed = 40,
    shard_options = grain.NoSharding()
)

In [8]:
dataloader = grain.DataLoader(
    data_source = test,
    sampler = sampler,
    operations = [ProcessStory(),
     grain.Batch(batch_size=32, drop_remainder=True)],
     worker_count = 0
)

In [10]:
for batch in dataloader:
    pprint(batch)
    break

array([[   2,    5, 5855, ...,    0,    0,    0],
       [   2,    5,  841, ...,    0,    0,    0],
       [   2,    5,  841, ...,    0,    0,    0],
       ...,
       [   2,    5,  326, ...,    0,    0,    0],
       [   2,    5,  282, ...,    0,    0,    0],
       [   2,    5,  841, ...,    0,    0,    0]],
      shape=(32, 1024), dtype=int32)


#### Building Token embedding and Positional embedding

##### testing the jax embedding

In [124]:
inputs = "Hello how are you"
tokens = tokenizer.encode(inputs).ids
jax_tokens = jnp.asarray([tokens])
jax_pos = jnp.arange(0, len(tokens))


vocab_size = tokenizer.get_vocab_size()
embed_dim = 2
maxlen = len(tokens)
print(maxlen)
rngs = nnx.Rngs(0)

token_embedding = nnx.Embed(num_embeddings = vocab_size, features = embed_dim, rngs = rngs)
positional_embedding = nnx.Embed(num_embeddings = maxlen, features = embed_dim, rngs = rngs)


5


In [115]:
token_embedding, positional_embedding

(Embed( # Param: 20,000 (80.0 KB)
   embedding=Param( # 20,000 (80.0 KB)
     value=Array(shape=(10000, 2), dtype=dtype('float32'))
   ),
   num_embeddings=10000,
   features=2,
   dtype=dtype('float32'),
   param_dtype=float32,
   promote_dtype=<function promote_dtype at 0xff69aa26d6c0>
 ),
 Embed( # Param: 10 (40 B)
   embedding=Param( # 10 (40 B)
     value=Array(shape=(5, 2), dtype=dtype('float32'))
   ),
   num_embeddings=5,
   features=2,
   dtype=dtype('float32'),
   param_dtype=float32,
   promote_dtype=<function promote_dtype at 0xff69aa26d6c0>
 ))

In [116]:
token_embedding(jax_tokens)

Array([[ 0.39644232,  0.5908411 ],
       [ 0.6713258 ,  0.41438612],
       [-0.775721  , -0.47760543],
       [ 0.5454083 ,  0.35636657],
       [ 0.52773136,  0.22185388]], dtype=float32)

In [117]:
positional_embedding(jax_pos)

Array([[-1.727077  , -1.4394435 ],
       [ 0.14534172, -0.24999774],
       [-0.538797  , -0.8333619 ],
       [-0.81191385,  0.21012793],
       [-0.9266888 ,  1.5062807 ]], dtype=float32)

##### Embedding block

Array([[   0,    1,    2, ..., 1021, 1022, 1023]], dtype=int32)

In [9]:
class EmbeddingBlock(nnx.Module):
    def __init__(self, vocab_size: int, maxlen: int, embed_dim: int, rngs: nnx.Rngs):
        self.token_embed = nnx.Embed(num_embeddings = vocab_size, features = embed_dim, rngs = rngs)
        self.pos_embed = nnx.Embed(num_embeddings = maxlen, features = embed_dim, rngs = rngs)

    def __call__(self, x: jax.Array) -> jax.Array:
        poisitons = jnp.arange(0, x.shape[1])[None, :] # create a 2D array
        position_embedding = self.pos_embed(poisitons)
        token_embedding = self.token_embed(x)
        return token_embedding + position_embedding

In [19]:
eb = EmbeddingBlock(vocab_size = 10000, maxlen = 1024, embed_dim = 768, rngs = nnx.Rngs(0))

In [20]:
for batch in dataloader:
    out = eb(batch)
    break

In [16]:
out.shape

(32, 1024, 768)

#### Transformer Block

In [11]:
# use this link: https://flax.readthedocs.io/en/stable/api_reference/flax.nnx/nn/attention.html#flax.nnx.MultiHeadAttention
class AttentionBlock(nnx.Module):
    def __init__(self, embed_dim: int, num_heads: int, d_ff: int, rngs: nnx.Rngs, rate: float = 0.1):
        self.mha = nnx.MultiHeadAttention(num_heads = num_heads, in_features = embed_dim, rngs = rngs)
        self.dropout1 = nnx.Dropout(rate = rate, rngs = rngs)
        self.layer_norm1 = nnx.LayerNorm(epsilon = 1e-6, num_features = embed_dim, rngs = rngs)
        self.mlp = nnx.Sequential(
            nnx.Linear(in_features = embed_dim, out_features = d_ff, rngs = rngs),
            nnx.relu,
            nnx.Linear(in_features = d_ff, out_features = embed_dim, rngs = rngs),
            nnx.Dropout(rate = rate, rngs = rngs))
        self.layer_norm2 = nnx.LayerNorm(epsilon = 1e-6, num_features = embed_dim, rngs = rngs)
    
    def __call__(self, inputs):
        causal_mask = nnx.make_causal_mask(inputs[..., 0], dtype=bool)
        attention_output = self.dropout1(self.mha(inputs_q = inputs, mask = causal_mask, decode = False))
        normalized = self.layer_norm1(inputs + attention_output)
        ffn_output = self.mlp(normalized)
        return self.layer_norm2(normalized + ffn_output)

        

In [52]:
ab = AttentionBlock(num_heads = 8, embed_dim = 768, d_ff = 768 * 4, rngs = nnx.Rngs(1))

In [53]:
for batch in dataloader:
    eb_out = eb(batch)
    print(eb_out[..., -1].shape)
    mask = nnx.make_causal_mask(eb_out[..., 0], dtype=bool)
    print(mask.shape)
    print(ab(eb_out).shape)
    break

(32, 1024)
(32, 1, 1024, 1024)
(32, 1024, 768)


#### GPT block

In [144]:
a = jnp.array(([3, 4, 5] + [2] + [0] * (10 - (3 + 1))))
a

Array([3, 4, 5, 2, 0, 0, 0, 0, 0, 0], dtype=int32)

In [ ]:
class GPT(nnx.Module):
    def __init__(self, vocab_size : int, maxlen : int, embed_dim: int, num_heads: int, d_ff: int, num_transformer_blocks: int, rngs: nnx.Rngs):
        self.embedding_block = EmbeddingBlock(vocab_size = vocab_size, maxlen = maxlen, embed_dim = embed_dim, rngs = rngs)
        self.transformer_blocks = nnx.Sequential(
            *[AttentionBlock(embed_dim = embed_dim, num_heads = num_heads, d_ff = d_ff, rngs = rngs)  for _ in range(num_transformer_blocks)]
        )
        self.output_layer = nnx.Linear(in_features = embed_dim, out_features = vocab_size, rngs = rngs)
    
    def __call__(self, inputs):
        x = self.embedding_block(inputs)
        x = self.transformer_blocks(x)
        return self.output_layer(x)
    
    def sample_from(self, logits, top_k = 10):
        logits, indices = jax.lax.top_k(logits, k = top_k) # https://docs.jax.dev/en/latest/_autosummary/jax.lax.top_k.html
        logits = jax.nn.softmax(logits) # https://docs.jax.dev/en/latest/_autosummary/jax.nn.softmax.html
        return jax.random.choice(jax.random.PRNGKey(1), indices, p = logits) # https://docs.jax.dev/en/latest/_autosummary/jax.random.choice.html

    @nnx.jit(donate_argnums=(1, ))
    def generate_step(self, padded_token, sample_index):
        logits = self(padded_token)
        next_token = self.sample_from(logits[0][sample_index])
        return next_token
        
    def generate_text(self, max_tokens, padded_tokens, sample_index):
        generated_tokens = []
        for i in range(max_tokens):
            sample_index += 1
            # print(f"Sample index: {sample_index}")
            next_token = self.generate_step(padded_tokens, sample_index)
            if next_token == tokenizer.encode("<|endoftext|>").ids[0]:
                break
            generated_tokens.append(next_token)
        return tokenizer.decode(generated_tokens)
    

In [37]:
def loss_fun(model, batch):
    logits = model(batch[0])
    loss = optax.softmax_cross_entropy_with_integer_labels(logits = logits, labels = batch[1])
    mask = (batch[1] != pad_id).astype(jnp.float32)
    masked_loss = (loss * mask).sum() / mask.sum()
    print(loss, mask, masked_loss)
    return masked_loss, logits

@nnx.jit(donate_argnums=(0, 1, 3))
def train_step(model: GPT, optimizer: nnx.Optimizer, metrics: nnx.MultiMetric, batch):
    grad_fun = nnx.value_and_grad(loss_fun, has_aux=True) # https://flax.readthedocs.io/en/stable/api_reference/flax.nnx/transforms.html#flax.nnx.value_and_grad
    (loss, logits), grad = grad_fun(model, batch)
    metrics.update(loss = loss, logits = logits, labels = batch[1]) # https://flax.readthedocs.io/en/stable/api_reference/flax.nnx/training/metrics.html
    optimizer.update(model, grad) # https://flax.readthedocs.io/en/stable/api_reference/flax.nnx/training/optimizer.html


In [34]:
vocab_size = 10_000
embed_dim = 768
maxlen = 1024
num_heads = 8
d_ff = embed_dim * 4
num_transformer_blocks = 4

num_epochs = 1
rngs = nnx.Rngs(0)

In [35]:
model = GPT(vocab_size, maxlen, embed_dim, num_heads, d_ff, num_transformer_blocks, rngs)
optimizer = nnx.Optimizer(model, optax.adam(1e-4), wrt=nnx.Param)
metrics = nnx.MultiMetric(loss = nnx.metrics.Average("loss"))
rng = jax.random.PRNGKey(0)

In [20]:
start_prompt = ["<|startoftext|> once up on a time"]
tokenizer_start_prompt = tokenizer.encode(start_prompt[0])
start_token, sample_index = jnp.asarray(tokenizer_start_prompt.ids)[None, :], sum(tokenizer_start_prompt.attention_mask)
# print(start_token, sample_index)
model.generate_text(10, start_token, sample_index)

'green cube  We are  was happy  to find  choo ar spoil reach it melon'

In [ ]:
print(model)

In [ ]:
for batch in dataloader:
    output = model(batch)
    print(output.shape)
    print(model.generate_text(10, jnp.asarray(tokenizer.encode("hello"))[None, :]))
    break

#### Training script

In [40]:
metric_history = {
    "train_loss": []
}

# https://docs.jax.dev/en/latest/_autosummary/jax.vmap.html
prep_target_batch = jax.vmap(
    lambda token: jnp.concatenate((token[1:], jnp.array([0]))) # we are creating target by shifting one token to the right, adding [0] for shape consistency
) # and using vmap cause we want to parallelize this lambda function over the batch dimension.

step = 0
for epoch in range(num_epochs):
    start_time = time.time()
    for batch in dataloader:
        # print(batch.shape)
        train_step(model, optimizer, metrics, (batch, prep_target_batch(batch)))
        if (step + 1) % 100 == 0:
            for metric, value in metrics.compute().items():
                metric_history[f"train_{metric}"].append(value)
            metrics.reset()
            elapsed_time = time.time() - start_time
            print(f"\n Step {step + 1}, Loss: {metric_history["train_loss"][-1]}, Elapsed Time: {elapsed_time:.2f} seconds")
            start_time = time.time()

            print("Generated text:")
            print(model.generate_text(100, start_token, sample_index))
        step += 1


 Step 100, Loss: 6.405951499938965, Elapsed Time: 61.34 seconds
Generated text:
y was  on sh w p th z c re on re sh she  v in her  c s c for it  i i al at i and  i " on ble ble sa c al ble special  ce se in her  c o  most  se and  it  c and  you  you  you  it  i al g c c sh them  ble you  in re .
The   adventure o  them  in her  she  ce in it  you  o  o  sh sad o  d. They  on it  ce on re  for on on its  ble se o   adventure it  on she  g t it  ce at them 

 Step 200, Loss: 6.137907981872559, Elapsed Time: 62.82 seconds
Generated text:
. She was three years old and  t st u d. He  y was  on v c nice  y,  z ss g ck z g end th  c special  and  z lesson al g she  d c " g z ck i ce old  and  ce al al ce re ble al al wonderful  ce g . It  re ce she  ss ss ce ce bit  sad g g she  Lily  al ss  adventure g she  ce g al sad re  adventure g d al old  i c sad c for she  she  ss re  ss for c al ce special  sad g smart al . It  ce re  sad

 Step 300, Loss: 5.911594867706299, Elapsed Time: 62.84 sec

In [41]:
import orbax.checkpoint as orbax
from pathlib import Path

state = nnx.state(model)
checkpoint_path = Path("checkpoints").resolve()
checkpointer = orbax.PyTreeCheckpointer()
checkpointer.save(checkpoint_path, args = orbax.args.PyTreeSave(state), force=True)

In [ ]:
print("Final generated text:")
tokenizer_start_prompt = tokenizer.encode("Once up on a time")
start_token, sample_index = jnp.asarray(tokenizer_start_prompt.ids)[None, :], sum(tokenizer_start_prompt.attention_mask)
# print(start_token, sample_index)
generated_text = model.generate_text(maxlen, start_token, sample_index)

In [111]:
generated_text

''

In [55]:
t = jnp.array([[1, 2, 3], [2, 3, 4]])

In [56]:
f = jax.vmap(lambda t: jnp.concatenate((t[1:], jnp.array([0]))))

In [57]:
f(t)

Array([[2, 3, 0],
       [3, 4, 0]], dtype=int32)

In [81]:
for batch in dataloader:
    print(batch.shape)
    input_b = jnp.stack(batch)
    output_b = prep_target_batch(input_b)
    print(input_b.shape, output_b.shape)
    print(input_b[0])
    print(output_b[0])
    break

(32, 1024)
(32, 1024) (32, 1024)
[  2   5 326 ...   0   0   0]
[   5  326 4506 ...    0    0    0]
